# 1805 loss-measurement audit

## tl;dr

No-go for one loss-based grade across all 22 entries. The inspected source supports a component table: 35/44 sides have some numerical personnel-loss observation, but only 6/22 entries have paired killed-and-wounded counts and 1/22 has paired prisoner counts. Unknowns stay null.

This notebook recomputes the central counts from the saved source records. It performs no historical model fit, source fetch or final-set evaluation.


## Context & Methods

Grain: one frozen entry side, A = France and its allies. Scope: all 22 land entries of `third-coalition 1805`, drawn from the existing development partition. One controlling family: Bodart (1908), entry pp. 363–370 and conventions pp. 45–47.

### Key Assumptions

- A source-reported component is not historically adjudicated truth.
- Only explicitly named all-ranks personnel categories enter component coverage. Equipment, officer and national subsets remain separate.
- Combined killed/wounded figures are not split. A dash or absent category is unknown, never an invented zero.
- A prisoner subset is not added to its aggregate. Surrender and conditional free withdrawal retain distinct meanings.
- This complete source group does not estimate corpus-wide feasibility or strategic value.


In [1]:
from pathlib import Path
import json
import sys
from fractions import Fraction

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "generalship/napoleonic_losses.py").is_file())
sys.path.insert(0, str(project_root))
from generalship import napoleonic_losses as audit

verified = audit.build(project_root)  # Hash, quotation, unit and grain checks; no fitting.
source = verified["observations"]
entries = source["entries"]
print("Verified source-local records; historical fits: 0; final entries used: 0")


Verified source-local records; historical fits: 0; final entries used: 0


## Data

Canonical observations: `data/napoleonic/loss-pilot-v1/observations.json`.
Frozen scope/protocol: the adjacent `scope.json` and `protocol.json`.
Controlling raw source: `data/raw/bodart-1908-v1/transcription.txt`, SHA-256 `bf12437b6d19226ce127ce5d81e59d20358a5b1999ffe00f5ac6a8f8c20ef986`.

The full source-linked table and all 19 percentage diagnostics are in `napoleonic-1805-loss-feasibility-v1.md`. Existing dossier disputes are linked, not newly adjudicated.


In [2]:
side_rows = [(e["battle_id"], name, side)
             for e in entries for name, side in e["sides"].items()]
assert len(entries) == 22
assert len(side_rows) == len({(battle, side) for battle, side, _ in side_rows}) == 44

def personnel_value(side, category):
    matching = [q["value"] for q in side["quantities"]
                if q["category"] == category and q["unit"] == "people"
                and q["population"] == "whole_side"]
    assert len(matching) <= 1
    return matching[0] if matching else None

categories = ("killed", "wounded", "killed_and_wounded", "captured", "missing", "reported_personnel_total")
available = lambda side: any(personnel_value(side, c) is not None for c in categories)
numeric_sides = sum(available(side) for _, _, side in side_rows)
numeric_pairs = sum(all(available(s) for s in e["sides"].values()) for e in entries)
assert (numeric_sides, numeric_pairs) == (35, 14)
print(f"Entries: {len(entries)}; side rows: {len(side_rows)}; duplicate side keys: 0")
print(f"Some numerical personnel observation: {numeric_sides}/44 sides, {numeric_pairs}/22 pairs")
print("Missing numerical side rows:", [(b, s) for b, s, row in side_rows if not available(row)])


Entries: 22; side rows: 44; duplicate side keys: 0
Some numerical personnel observation: 35/44 sides, 14/22 pairs
Missing numerical side rows: [('B364a', 'A'), ('B364c', 'A'), ('B364d', 'A'), ('B365b', 'A'), ('B365d', 'A'), ('B366a', 'A'), ('B368b', 'A'), ('B368d', 'A'), ('B368d', 'B')]


## Results

### Component coverage

The table distinguishes numerical presence from a shared category. Reported totals can have different composition; their co-presence is not proof of comparable attrition. This small audit uses exact-value tables rather than a chart because the definitions and denominators drive the decision.


In [3]:
computed = {}
print(f'{"All-ranks category":27} {"Sides / 44":>12} {"Pairs / 22":>12}')
for category in categories:
    side_n = sum(personnel_value(row, category) is not None for _, _, row in side_rows)
    pair_ids = [e["battle_id"] for e in entries
                if all(personnel_value(row, category) is not None for row in e["sides"].values())]
    computed[category] = (side_n, pair_ids)
    expected = verified["profile"]["category_coverage"][category]
    assert side_n == expected["sides"] and pair_ids == expected["pair_ids"]
    print(f'{category:27} {side_n:>12} {len(pair_ids):>12}')
assert computed["killed_and_wounded"][0] == 16
assert len(computed["killed_and_wounded"][1]) == 6
assert len(computed["captured"][1]) == 1
assert all(len(pair_ids) < 22 for _, pair_ids in computed.values())
print("Paired killed/wounded IDs:", computed["killed_and_wounded"][1])
print("Decision: no single defined component covers the whole fixed group.")


All-ranks category            Sides / 44   Pairs / 22
killed                                 0            0
wounded                                0            0
killed_and_wounded                    16            6
captured                              15            1
missing                                0            0
reported_personnel_total              28           10
Paired killed/wounded IDs: ['B363a', 'B363b', 'B365c', 'B367a', 'B368c', 'B369c']
Decision: no single defined component covers the whole fixed group.


### Nested quantities and percentage arithmetic

Hollabrunn's prisoner figure is within its reported total. The printed percentages are compared with the printed counts without selecting corrected historical figures. The three examples below are displayed for readability; all 19 diagnostics are recomputed and checked.


In [4]:
indexed = {e["battle_id"]: e for e in entries}
hollabrunn = indexed["B369a"]["sides"]["B"]
assert personnel_value(hollabrunn, "reported_personnel_total") == 3000
assert personnel_value(hollabrunn, "captured") == 1800
assert personnel_value(hollabrunn, "killed_and_wounded") is None
print("Hollabrunn B: total ca. 3,000 includes 1,800 prisoners; no 4,800 total or 1,200 killed/wounded is created.")

examples = {("B363c", "A", "reported_personnel_total"),
            ("B368c", "A", "reported_personnel_total"),
            ("B367a", "A", "reported_personnel_total")}
for row in verified["profile"]["percent_checks"]:
    ratio = Fraction(100 * row["numerator_people"], row["denominator_people"])
    difference = Fraction(str(row["printed_percent"])) - ratio
    assert round(float(ratio), 6) == row["implied_percent"]
    assert round(float(difference), 6) == row["printed_minus_implied_pp"]
    if (row["battle_id"], row["side"], row["category"]) in examples:
        print(f'{row["battle_id"]} {row["side"]}: printed {row["printed_percent"]:g}%; '
              f'count ratio {float(ratio):.6f}%; difference {float(difference):+.6f} pp')
assert len(verified["profile"]["percent_checks"]) == 19
assert all(row["difference_people"] == 0 for row in verified["profile"]["source_sum_checks"])
print("Checked 19 percentage diagnostics and 10 printed personnel sums; all source figures retained.")


Hollabrunn B: total ca. 3,000 includes 1,800 prisoners; no 4,800 total or 1,200 killed/wounded is created.
B363c A: printed 32%; count ratio 25.000000%; difference +7.000000 pp
B367a A: printed 18.4%; count ratio 17.391304%; difference +1.008696 pp
B368c A: printed 88%; count ratio 82.500000%; difference +5.500000 pp
Checked 19 percentage diagnostics and 10 printed personnel sums; all source figures retained.


## Takeaways

The source-local component table is supportable. A complete, common loss-based grade for these 22 entries is not. Incomplete category coverage, conditional release and different loss periods remain substantive limits. Loss components alone do not measure strategic value or commander contribution.

The pilot stops here. No model input, original source, dossier, benchmark or prior result is changed; all 52 final entries remain reserved. The companion is an arithmetic check, not an independent historical review.
